In [1]:
using LinearAlgebra, BenchmarkTools, PolynomialRoots, StaticArrays, DataStructures

# Fully-split velocity Lagrangian PDMP

(Add comments from the Overleaf document about the splitting and dynamics)

## Ricatti equation and the velocity flow
The fully split velocity satisfies the Ricatti equation

$du/dt = a u^2 + b u + c$

for some $a,b,c$. This admits the solution

$u(t) = (\kappa * \tan(\kappa*(t+t_0))-(b/2))/a$

where $\kappa = \sqrt{4ac-b^2}/2$

In [2]:
riccati(t, a, b, c; t0 = 0.0) = ricatti_κ(t, a, b, sqrt(a*c-(b/2)^2), t0 = t0)
ricatti_κ(t, a, b, κ; t0 = 0.0) = (κ * tan(κ*(t+t0))-(b/2))/a

@benchmark ricatti_κ($0.2, $1.0, $2.0, $3.0)

BenchmarkTools.Trial: 10000 samples with 998 evaluations per sample.
 Range (min … max):  16.132 ns … 266.132 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     17.735 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   20.540 ns ±   9.572 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▁ █  ▄                                                        
  ███▆▅█▁▂▃▂▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  16.1 ns         Histogram: frequency by time         40.4 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

## Rate integrals and rates
### Exact methods
The rates are given by some expression

$\lambda_{IJ} = [\rho_{IJ}]^+ = [A^I - A^J]^+/n$

and the total rate in state $I$ is simply the sum $\lambda_I = \sum_J\lambda_{IJ}$. The $A^I$ are cubic in $u(t)$ so each $\lambda_{IJ}$ is some cubic polynomial in $u(t)$. We can explicitly integrate each $\lambda_{IJ}$ *if* we know that it is positive. Thus we solve $\rho_{IJ}(u) = 0$ for all $I,J$ and order the individual solutions $u_1, u_2, \ldots$ such that $u_i < u_{i+1}$ if $du/dt > 0$, and $u_i> u_{i+1}$ else (Note that $du/dt$ is actually identically positive or negative for all $t$ we will consider, since $u(t)$ 'blows up' in finite time). This partitions the velocity space $U$ into sets over which the signs of all $\rho_{IJ}$ are constant and over these the sum $\sum_J \lambda_{IJ}$ can thus be computed.

In [3]:
pol = [9,3,-7,1]
@benchmark roots($pol)

BenchmarkTools.Trial: 10000 samples with 195 evaluations per sample.
 Range (min … max):  493.333 ns … 170.522 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     555.385 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   754.130 ns ±   2.210 μs  ┊ GC (mean ± σ):  6.95% ± 5.28%

  ▇█▆▅▅▄▃▃▃▃▂▁▁▁▃▄▄▄▃▂                                          ▂
  ███████████████████████▇▇▇▇▇▇████▆▅▅▆▆▅▅▅▆▆▆▆▅▄▄▅▄▅▅▅▄▅▁▄▅▁▄▅ █
  493 ns        Histogram: log(frequency) by time       2.14 μs <

 Memory estimate: 496 bytes, allocs estimate: 8.

In [4]:
"""
    real_roots(b, c, d; verbose=false)::Tuple

Finds the real roots of the cubic polynomial x^3 + b x^2 +c x + d.
"""
function real_roots(b, c, d; verbose=false)::Tuple
    Δ = b^2 - 3*c
    μ = (2*(b^3)) - (9*b*c) + (27*d)

    if iszero(Δ) && iszero(μ) #Should check vs threshold.
        return (-b/3,) 
    end

    #Optimize: Add special statement for when Δ = 0 (within threshold.)
    L = (μ^2)-(4*Δ^3)
    if L < 0 
        verbose ? println("L < 0") : nothing
        z = (μ + sqrt(-L)*im) #2z = ...,  but we only use the angle for z
        r2 = cbrt((μ^2 - L)/4)
        if r2 ≈ Δ #This must be handled more carefully!
            verbose ? println("r2 = Δ, difference: $(r2-Δ)") : nothing

            (s, c) = sincos(angle(z)/3) #can be optimized/altered to use the 'tan-formula' for the 3xReal root cubic
            k = 2*sqrt(r2)
            return (b .+ (k.* (c, (-c + (sqrt(3)*s))/2, (-c - (sqrt(3)*s))/2))) ./(-3)
        else
            verbose ? println("r2 ≠ Δ, difference: $(r2-Δ)") : nothing
            if μ ≤ 0
                verbose ? println("μ ≤ 0") : nothing
                return (b + sqrt(r2)*(1+(Δ/(r2))),) ./(-3)
            else
                verbose ? println("μ > 0") : nothing
                return (b - sqrt(r2)*(1+(Δ/(r2))),)./(-3)
            end
        end
    elseif L ≥ 0
        verbose ? println("L ≥ 0, L: $L") : nothing
        root = sqrt(L)
        if μ > 0
            z = (μ + root)/2
        else
            z = (μ - root)/2
        end
            
        C = cbrt(z)
        r2 = C^2
        if r2 ≈ Δ #This must be handled more carefully! We should check roots (amounts to a single computation) 
            verbose ? println("r2 = Δ, difference: $(r2-Δ)") : nothing
            return (b+2*C, b-C) ./ (-3)
        else
            verbose ? println("r2 ≠ Δ, difference: $(r2-Δ)") : nothing
            return (b + C*(1+(Δ/r2)),) ./(-3)
        end
    end
end

real_roots(a,b,c,d) = real_roots(b/a, c/a, d/a)

real_roots (generic function with 2 methods)

Thus we have established a pretty decent root-finder. Let's look at its performance:

In [5]:
bm_roots_1R = @benchmark real_roots($2.0, $(-3.0), $9.0) #One real root

BenchmarkTools.Trial: 10000 samples with 983 evaluations per sample.
 Range (min … max):  54.832 ns … 804.171 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     58.494 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   64.838 ns ±  18.734 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

   ██▇▆▅▂▂▃▄▂▁     ▁  ▂▃▄▃▂▂  ▁  ▁                             ▂
  ▆█████████████▇▇▇█▇▇██████████▆█▇█▇▆▇▆▅▆▆▅▅▅▄▄▃▂▃▄▅▄▄▃▃▄▄▄▄▄ █
  54.8 ns       Histogram: log(frequency) by time       128 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [6]:
bm_roots_3R = @benchmark real_roots($(-7.0), $(3.0), $9.0) #Three real roots

BenchmarkTools.Trial: 10000 samples with 961 evaluations per sample.
 Range (min … max):   87.825 ns …  1.351 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):      96.358 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   112.391 ns ± 45.279 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

   █▃                                                           
  ▆██▆▄▃▂▂▁▁▁▁▁▁▁▁▁▂▂▃▅▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  87.8 ns         Histogram: frequency by time          259 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

This is something like $\sim 10-15$ times faster than the one defined above. As a rough measure of performance then we expect something like $\sim 80 N ns$ to compute the partition of the time interval for a velocity update. For e.g. the 20-dimensional Twin Peaks scenario $N = 21$ so we get something akin to $\sim 1.6 \mu s$ of work. Naturally there may be other f Of course, if we wanted to we could parallellize this particular task, but the overhead for such an endeavour would probably be prohibitive. 

#### Partitioning the time intervals

It shall become convenient to define $A^I(u) = \sum_j A^I_j u^j $ and $\rho^{IJ} = \sum_j\rho^{IJ}_ju_j$.

In [7]:
real_roots(X::SVector{4, Float64}) = real_roots(X[1], X[2], X[3], X[4])

real_roots (generic function with 3 methods)

In [15]:
A1=As[1]
@benchmark real_roots($A1) 

BenchmarkTools.Trial: 10000 samples with 980 evaluations per sample.
 Range (min … max):  60.204 ns …  1.096 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     63.673 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   71.469 ns ± 25.758 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▆█▅▅▂▃▄▂▁▁▁▂▁    ▂▄▂  ▁                                     ▂
  ████████████████████████▇▇▇▇▇▇▆▇▇▆▆▆▆▅▅▅▅▅▄▅▅▅▄▅▄▄▄▅▅▁▅▄▄▅▆ █
  60.2 ns      Histogram: log(frequency) by time       167 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [87]:
function partition_by_roots!(T::BinaryMinHeap{Float64}, ρ_IJs::Vector{SVector{4, Float64}}; split_dim = length(As), start_time = 0.0) #could be $MVector
    empty!(T)
    push!(T, start_time)
    for J in eachindex(ρ_IJs)
        roots = real_roots(ρ_IJs[J])
        for root in roots
            if root > start_time
                push!(T, root)
            end
        end
    end
    return T
end            

partition_by_roots! (generic function with 1 method)

In [88]:
D = 21
As = SVector{4, Float64}[]
for i in 1:D
    push!(As, @SVector rand(4))
end
T = BinaryMinHeap{Float64}()

BinaryMinHeap{Float64}(Base.Order.ForwardOrdering(), Float64[])

In [89]:
@benchmark partition_by_roots!($T, $As)

BenchmarkTools.Trial: 10000 samples with 10 evaluations per sample.
 Range (min … max):  1.340 μs …  18.400 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     1.390 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   1.498 μs ± 476.565 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▅█▅▇▄▁                  ▁▂▂▁▁      ▁▁▁                      ▂
  ██████▇▅▇▇█▆▆▅▄▆▅▅▆▆▆▆▆███████▇▆▇█▇███▇▅▆▅▄▅▅▃▄▁▃▁▅▄▄▅▃▄▄▅▆ █
  1.34 μs      Histogram: log(frequency) by time      2.87 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

Indeed, we got roughly to $1.6\mu s$ of computation time.

#### The rate integrals 
The rates need to be integrated, that is we integrate

$\lambda = A u^3 + B u^2 + C u + D = \bar{A} \cdot \bar{U}$

over time (on the intervals discussed above). Each integral

$M_n = \int u(t)^n dt$

can be analytically computed. We expand in the $\tan(\kappa (t+t_0))$ after a transformation $s = \kappa(t+t_0)$ so that

$dt = ds/\kappa$

and (by abuse of notation)

$u(s) = (\kappa \tan(s) -(b/2))/a$

whence

$M_n(s) = \frac{1}{a^n\kappa} \sum_{j=0}^n \binom{n}{j} (-b/2)^{n-j} \int (\kappa \tan(s))^{j}ds = \frac{1}{a^n\kappa} \sum_{j=0}^n \binom{n}{j} \kappa^{j}(-b/2)^{n-j} L_{j} = \frac{1}{a^n\kappa}\bar{M_n} \cdot \bar{L}$

with each $L_j$ corresponding to a $\tan(x)^j$ integral. It can be shown that

1) $L_0 = s$
2) $L_1 = -\log(\cos(s))$
3) $L_2 = (\tan(s)-s)$
3) $L_3 = (\frac{1}{2\cos^2(s)} + \log(\cos(s)))$

Obviously there are some issues with the values - they can be divergent, complex and otherwise problematic. However, we shall always have $s$ in specific domains, and the integrals shall only be integrated over domains for which $\lambda \geq 0$.

Of course we shall typically evaluate this at many distinct "times" $s$, which alters the vector $L$. Thus it makes sense to use a matrix $M$ with the distinct $M_i$ as rows so that $\bar{U} = M \bar{L}$

In [ ]:
function L_tuple(s::Float64)
    c = cos(s)
    lc = log(c)
    return (s, -lc, tan(s) -s , lc + (1. /(2*(c^2))))
end

L_tuple (generic function with 1 method)

In [84]:
function M_matrix!(M::MMatrix{4,4, Float64, 16}, a, b, κ)
    for i in 1:4
        for j in 1:i
            M[i, j] = binomial(i, j) * ((-b / 2.)^j) * (κ^(i-j))
        end
        factor = (κ * (a^i))
        @views M[i,:] ./= factor
    end
    return M 
end

M_matrix! (generic function with 4 methods)

In [85]:
M = @MMatrix zeros(4,4)
M_matrix!(M, rand(), randn(), rand())

4×4 MMatrix{4, 4, Float64, 16} with indices SOneTo(4)×SOneTo(4):
  -2.70791   0.0        0.0       0.0
  -5.33126   4.71954    0.0       0.0
  -7.87201  13.9375    -8.22554   0.0
 -10.3321   27.4398   -32.3884   14.336

In [86]:
@benchmark M_matrix!($M, $a, $b, $κ)

BenchmarkTools.Trial: 10000 samples with 421 evaluations per sample.
 Range (min … max):  236.817 ns …  1.196 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     251.069 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   280.751 ns ± 74.558 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▅█▇▃▃▄▂▁▂▃▃▄▁▄▁▁▁▁                                           ▂
  █████████████████████▇▇▇▇▇▇▆▆▇▆▆▆▅▆▅▇█▇▇███████▇▆▆▅▆▆▆▅▆▄▅▅▄ █
  237 ns        Histogram: log(frequency) by time       576 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [ ]:
function compute_rate_integrals(ρ_IJs::Vector{SVector{4, Float64}}, M::MMatrix, initial_values::Vector{Float64}, s_final::Float64)
    